# ElevenLabs

A refresher on **ElevenLabs** — the cloud AI-voice platform whose **Text-to-Speech** API turns a string into remarkably natural, expressive speech, plus **voice cloning**, **speech-to-speech**, **dubbing**, **sound effects**, and a **Conversational AI** (voice-agent) stack. It is the quality benchmark for hosted TTS: zero model ops, a few lines of code, billed per character. You hold an API key and call an endpoint; the models run on their GPUs.

**Domain:** Speech & Audio  ·  **from study list**  ·  **runnable:** yes  ·  _the no-key cells (cost accounting + REST request shape) run on CPU with stdlib only; the real TTS call is gated behind `ELEVENLABS_API_KEY`_

## 1. What & Why

**What it is.** ElevenLabs is a hosted voice-AI provider. The headline product is **Text-to-Speech**: `POST` some text plus a `voice_id` and a `model_id`, get back audio (MP3 or PCM). Around it sits a suite that all shares the same voices and key — **Instant/Professional Voice Cloning**, a community **Voice Library**, **Speech-to-Speech** (re-voice an existing recording), **Dubbing** (translate + lip-aligned re-voice video), **Sound Effects** (text→SFX), **Studio** (long-form narration projects), **Scribe** speech-to-text, and **Conversational AI** for low-latency voice agents.

**The problem it solves.** Self-hosted TTS (Coqui, Piper, StyleTTS2) means GPUs, model downloads, vocoder tuning, and quality that still trails the best. ElevenLabs removes all of that: state-of-the-art naturalness and prosody, ~30 languages, instant voice cloning from a short sample, and sub-100 ms streaming — behind one REST call. You trade money and a network round-trip for top-tier quality with zero ML ops.

**When to reach for it.** Audiobooks, narration, ads, and video voiceover where naturalness sells the product; **voice cloning** a brand or character voice; real-time **conversational agents** (use the Flash model for latency); dubbing/localization; rapid prototyping where you don't want to stand up a model. It is the default when quality matters more than per-character cost or data residency.

**When not to.** High-volume or cost-sensitive batch synthesis (per-character billing adds up fast); strict data-residency / offline / air-gapped requirements (audio leaves your box); fully deterministic output (it's a hosted model that can drift between versions); or embedded/on-device use — reach for **Piper** or **Coqui** there. See §7.

## 2. Mental Model

**ElevenLabs is a vending machine for voices. You don't own or run the model — you put in a key, pick a *voice* (which slot) and a *model* (quality vs. speed tradeoff), feed in text, and audio comes out. You pay per character of text you push through.**

```
                          your machine                          ElevenLabs cloud (their GPUs)
 ┌───────┐   xi-api-key   ┌─────────────────────────┐  HTTPS   ┌──────────────────────────────┐
 │ text  │ ─────────────▶ │ POST /v1/text-to-speech │ ───────▶ │  model_id  (multilingual_v2 / │
 │ +voice│                │      /{voice_id}        │          │   turbo / flash / v3)         │
 │ +model│                │  body: text, model_id,  │          │  + voice_id (timbre/identity) │
 └───────┘                │  voice_settings,        │          │  + voice_settings (delivery)  │
                          │  output_format          │ ◀─────── │  ── synthesize ──▶ audio bytes│
                          └─────────────────────────┘  mp3/pcm └──────────────────────────────┘
```

Three knobs decide everything, and they're orthogonal:

1. **`voice_id` = *who* speaks** (timbre/identity) — a library voice, a default, or your clone.
2. **`model_id` = *how good / how fast*** — `multilingual_v2` (best quality), `turbo_v2_5` (balanced), `flash_v2_5` (~75 ms, for real-time agents), `v3` (most expressive, audio tags). This is the quality⇄latency dial.
3. **`voice_settings` = *how it's delivered*** — `stability` (consistent vs. expressive), `similarity_boost`, `style`, `use_speaker_boost`. Same voice, different performance.

You never touch weights, vocoders, or sample rates beyond picking an `output_format`.

## 3. Key Concepts

- **`voice_id`.** Stable identifier for a voice — a default (e.g. *Rachel* `21m00Tcm4TlvDq8ikWAM`), a Voice-Library voice you've added, or your own clone. It selects *identity/timbre*, independent of the model.
- **Models (`model_id`).** The quality⇄latency dial: **`eleven_multilingual_v2`** (highest quality, ~29 languages, the workhorse); **`eleven_turbo_v2_5`** (good quality, low latency); **`eleven_flash_v2_5`** (ultra-low latency ~75 ms, for conversational/real-time); **`eleven_v3`** (most expressive, supports inline *audio tags* like `[whispers]`/`[laughs]`). Flash/Turbo bill at a discount (see credits).
- **`voice_settings`.** Performance controls. **`stability`** (0–1): low = more emotional/variable but can wobble; high = consistent but flatter. **`similarity_boost`**: how tightly to adhere to the original voice. **`style`**: style exaggeration (v2 models; adds latency, can reduce stability). **`use_speaker_boost`**: extra fidelity to the source voice.
- **Voice cloning.** **Instant Voice Cloning (IVC)** — a clone from ~1–3 minutes of audio, ready immediately. **Professional Voice Cloning (PVC)** — a fine-tuned, higher-fidelity model from 30 min–3 hr of clean audio; trains for hours. Both need consent.
- **Credits & billing.** Usage is metered in **credits ≈ characters**. `multilingual_v2` costs **1 credit/char**; **Flash/Turbo cost 0.5 credit/char**. Plans (Free → Starter → Creator → Pro → Scale → Business) grant a monthly credit quota and a concurrency limit.
- **`output_format`.** Codec + sample rate + bitrate, e.g. `mp3_44100_128` (default), `pcm_24000`/`pcm_16000` (raw, for further processing), `ulaw_8000` (telephony/Twilio). Higher-quality MP3 and PCM formats require higher plans.
- **Streaming & WebSockets.** `convert()` returns the whole clip; `stream()` yields audio chunks as they're generated (lower time-to-first-byte); a WebSocket API streams text→audio incrementally for live agents.
- **Pronunciation dictionaries.** Upload IPA/CMU phoneme rules (or aliases) to force correct pronunciation of names, jargon, and acronyms — the structured fix for mispronunciations.

## 4. Setup

```bash
pip install elevenlabs        # official Python SDK (or just call the REST API with `requests`)

export ELEVENLABS_API_KEY="sk_..."   # create a key in the dashboard -> Profile -> API Keys
```

Get a key from the [ElevenLabs dashboard](https://elevenlabs.io/app/settings/api-keys). The **Free** tier gives ~10k credits/month and is enough to try the API (clips carry attribution). No model download and nothing runs locally — all synthesis happens on their servers, so every real call needs network + key and consumes credits.

The runnable cells below stay **self-contained and key-free**: Example 1 reproduces the **credit/cost math** and Example 2 builds the exact **REST request** (headers + JSON body) you'd send — both pure stdlib, no network. Example 3 makes a **real TTS call** but is gated behind `ELEVENLABS_API_KEY`, so the notebook always executes top-to-bottom.

In [ ]:
import sys, json

print(f"python {sys.version.split()[0]}")
print("ElevenLabs = hosted TTS: pick voice_id (who) + model_id (quality/latency) + voice_settings (delivery).")
print("The next two cells run with NO key and NO network: credit math, then the REST request shape.")

## 5. Worked Examples

### Example 1 — Credit accounting: estimate cost before you synthesize

ElevenLabs bills in **credits ≈ characters**, but the rate depends on the model: `multilingual_v2`
is **1 credit/char** while **Flash/Turbo are 0.5 credit/char**. Before pushing a script you want to
know how many credits it burns and whether it fits your monthly quota. This is the single most useful
back-of-envelope for the platform — pure Python, no key needed.

In [ ]:
# Credit rate per character, by model (Flash/Turbo are half-price).
RATE = {
    "eleven_multilingual_v2": 1.0,
    "eleven_v3":              1.0,
    "eleven_turbo_v2_5":      0.5,
    "eleven_flash_v2_5":      0.5,
}

# Approx. monthly credit quotas by plan (credits == characters at rate 1.0).
PLAN_CREDITS = {"free": 10_000, "starter": 30_000, "creator": 100_000,
                "pro": 500_000, "scale": 2_000_000}

def credits(text, model_id):
    return len(text) * RATE[model_id]

script = ("Welcome to the show. " * 50).strip()   # ~1000-char sample script
print(f"script length: {len(script)} characters\n")

print(f"{'model':<24}{'credits':>10}{'fits free?':>14}")
for m in RATE:
    c = credits(script, model_id=m)
    fits = "yes" if c <= PLAN_CREDITS["free"] else "no"
    print(f"{m:<24}{c:>10.0f}{fits:>14}")

# How many such clips before each plan's monthly quota is exhausted (multilingual_v2)?
print("\nclips/month at multilingual_v2 (1 credit/char):")
for plan, quota in PLAN_CREDITS.items():
    print(f"  {plan:<9} {quota // int(credits(script, 'eleven_multilingual_v2')):>5} clips")

### Example 2 — Build the exact REST request (headers + JSON body)

The SDK is a thin wrapper over one HTTP call. Knowing the raw contract means you can debug, port to
any language, or use plain `requests`. We construct the headers, the JSON body (text + `model_id` +
`voice_settings`), and the `output_format` — then print the request and the equivalent `curl`. No
network call is made, so nothing is sent and no credits are spent.

In [ ]:
VOICE_ID = "21m00Tcm4TlvDq8ikWAM"   # "Rachel", a default voice
BASE = "https://api.elevenlabs.io/v1/text-to-speech"

headers = {
    "xi-api-key": "$ELEVENLABS_API_KEY",   # placeholder; never hard-code the real key
    "Content-Type": "application/json",
    "Accept": "audio/mpeg",
}

body = {
    "text": "ElevenLabs turns this sentence into natural speech.",
    "model_id": "eleven_multilingual_v2",
    "voice_settings": {
        "stability": 0.5,          # lower = more expressive/variable, higher = more consistent
        "similarity_boost": 0.75,  # adherence to the original voice
        "style": 0.0,              # style exaggeration (v2 models); >0 adds latency
        "use_speaker_boost": True,
    },
}

output_format = "mp3_44100_128"
url = f"{BASE}/{VOICE_ID}?output_format={output_format}"

print("POST", url)
for k, v in headers.items():
    print(f"  {k}: {v}")
print("\nbody:")
print(json.dumps(body, indent=2))

print("\n# equivalent curl:")
print(f"""curl -X POST '{url}' \\
  -H 'xi-api-key: $ELEVENLABS_API_KEY' \\
  -H 'Content-Type: application/json' \\
  -d '{json.dumps(body)}' \\
  --output out.mp3""")

### Example 3 — A real TTS call with the Python SDK (gated)

With the `elevenlabs` package installed and `ELEVENLABS_API_KEY` set, synthesis is a few lines.
This makes a real network call and **spends credits**, so it's gated behind the env var. Either way
the cell prints the canonical call shapes — convert-to-bytes, low-latency streaming, and listing
your available voices.

In [ ]:
import os

if os.getenv("ELEVENLABS_API_KEY"):
    from elevenlabs.client import ElevenLabs
    from elevenlabs import save

    client = ElevenLabs(api_key=os.environ["ELEVENLABS_API_KEY"])

    # Full clip as bytes -> save to file.
    audio = client.text_to_speech.convert(
        voice_id="21m00Tcm4TlvDq8ikWAM",          # Rachel
        model_id="eleven_multilingual_v2",
        text="Hello from ElevenLabs.",
        output_format="mp3_44100_128",
    )
    save(audio, "out.mp3")
    print("wrote out.mp3")

    # First few of your available voices.
    voices = client.voices.get_all().voices
    print("voices:", [(v.name, v.voice_id) for v in voices[:3]])
else:
    print("Set ELEVENLABS_API_KEY (and `pip install elevenlabs`) to synthesize for real.\n")
    print("from elevenlabs.client import ElevenLabs")
    print("from elevenlabs import play, save")
    print("client = ElevenLabs(api_key=...)\n")
    print("# full clip -> bytes")
    print('audio = client.text_to_speech.convert(')
    print('    voice_id="21m00Tcm4TlvDq8ikWAM", model_id="eleven_multilingual_v2",')
    print('    text="Hello.", output_format="mp3_44100_128")')
    print('save(audio, "out.mp3")   # or play(audio)\n')
    print("# low-latency streaming (chunks as generated) -- use flash for real-time agents")
    print('stream = client.text_to_speech.stream(')
    print('    voice_id="21m00Tcm4TlvDq8ikWAM", model_id="eleven_flash_v2_5", text="Hi.")')
    print('for chunk in stream: ...   # forward to speaker / socket\n')
    print("# list your voices")
    print("client.voices.get_all().voices")

## 6. Gotchas & Pitfalls

- **`stability` is a tradeoff, not "better is higher".** High stability → consistent but flat/monotone; low → expressive but can wobble, mispronounce, or hallucinate on tricky text. Tune per use case (narration high-ish, character work lower).
- **Per-character billing surprises.** Credits ≈ characters; long scripts and retries add up fast, and `style > 0` plus high-quality formats cost more (latency too). Estimate first (Example 1) and prefer **Flash/Turbo** (0.5 credit/char) when their quality suffices.
- **Pick the model for latency, not just quality.** `multilingual_v2` is great but slow for live use. Real-time agents need **`flash_v2_5`** (~75 ms); `turbo_v2_5` is the middle ground. Wrong model = sluggish conversation.
- **Determinism is best-effort.** It's a hosted model; output can vary run-to-run and **change when ElevenLabs updates a model version**. Pin `model_id`, pass a `seed` where supported, and don't assume byte-identical audio.
- **Concurrency limits per plan.** Each tier caps simultaneous requests; batch jobs hit `429`/quota errors. Throttle, back off, and check your plan's concurrency before parallelizing.
- **Pronunciation needs help for names/jargon.** Normalization handles common cases but mangles unusual names, acronyms, and numbers in context. Use **pronunciation dictionaries** (IPA/CMU or aliases) rather than hacking spelling.
- **Output format must match downstream.** Telephony wants `ulaw_8000`; further DSP wants raw `pcm_*`; web wants `mp3_*`. Higher-quality MP3/PCM formats are gated behind higher plans — a silent feature limit, not a bug.
- **Voice cloning needs clean audio and consent.** IVC/PVC quality tracks the reference: clean, single-speaker, consistent. Cloning a real person's voice without permission violates the terms (and likely the law).
- **Request length limits.** Single TTS requests cap text length (a few thousand chars); for books/articles split into chunks (or use **Studio**) and stitch, keeping voice/settings consistent across chunks.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs ElevenLabs |
|---|---|---|
| **ElevenLabs** | Best-in-class naturalness, easy voice cloning, expressive v3, real-time agents | Per-character cost, cloud-only (audio leaves box), non-deterministic |
| **Azure / Google / Amazon Polly TTS** | Enterprise scale, SLAs, SSML, huge voice catalog, compliance | Generally less natural/expressive; more config; still paid cloud |
| **OpenAI TTS** | Simple, cheap, good quality inside the OpenAI stack | Fewer voices, no real voice cloning, less control over delivery |
| **Coqui TTS (XTTS-v2)** | Self-hosted open cloning + multilingual, no per-use bill | You run GPUs/ops; quality trails ElevenLabs; community-maintained |
| **Piper** | Fast, tiny, fully offline/on-device (Raspberry Pi, Home Assistant) | Lower naturalness; no zero-shot cloning; fewer voices |
| **StyleTTS2 / Bark / Tortoise** | Open expressive research models, full local control | Heavier setup, slower, less of a unified product/SLA |

**Rule of thumb:** choose **ElevenLabs when quality, expressiveness, or fast voice cloning matter and you can pay per character and send audio to the cloud** — it's the naturalness leader with the least ops. Drop to **Azure/Google/Polly** for enterprise compliance and scale, to **Coqui/Piper** when you need self-hosted/offline or high-volume cost control, and to **OpenAI TTS** when you just want decent voice cheaply inside that ecosystem.

## 8. Resources

- **ElevenLabs Docs** — products, guides, concepts: https://elevenlabs.io/docs
- **Text-to-Speech API reference** — endpoint, params, formats: https://elevenlabs.io/docs/api-reference/text-to-speech/convert
- **Models overview** — multilingual_v2 vs turbo vs flash vs v3: https://elevenlabs.io/docs/models
- **Python SDK (GitHub)** — install, examples, streaming: https://github.com/elevenlabs/elevenlabs-python
- **Voice settings guide** — stability / similarity / style explained: https://elevenlabs.io/docs/best-practices/prompting
- **Pricing & credits** — plans, quotas, concurrency: https://elevenlabs.io/pricing

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
RATE = {}
PLAN_CREDITS = {}


def credits(text, model_id):
    ...


def build_request(voice_id, text, model_id, **kwargs):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE